# SFT Training - Qwen 0.5B on GSM8K

In [1]:
import torch
import os
import mlflow
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer
from datasets import load_dataset

os.environ["MLFLOW_TRACKING_URI"] = "http://mlflow:5000"
mlflow.set_experiment("sft_qwen_gsm8k")

2026/02/18 04:58:50 INFO mlflow.tracking.fluent: Experiment with name 'sft_qwen_gsm8k' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/910887090828468506', creation_time=1771390730825, experiment_id='910887090828468506', last_update_time=1771390730825, lifecycle_stage='active', name='sft_qwen_gsm8k', tags={}>

In [2]:
# Load model and tokenizer
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [3]:
# Load GSM8K dataset
dataset = load_dataset("openai/gsm8k", "main", split="train")
print(f"Dataset size: {len(dataset)}")
print(f"Example: {dataset[0]}")

Dataset size: 7473
Example: {'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


In [4]:
def format_prompt(example):
    return f"Question: {example['question']}\nAnswer: {example['answer']}"

In [5]:
# Training arguments
training_args = TrainingArguments(
    output_dir="/app/models/sft_qwen",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    report_to="mlflow",
    run_name="sft_qwen_gsm8k",
)

In [6]:
# Initialize trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    formatting_func=format_prompt,
)

In [7]:
# Train
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


2026/02/18 04:58:57 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



Step,Training Loss
10,0.494776
20,0.431358
30,0.392504
40,0.380320
50,0.396378
60,0.404276
70,0.364976
80,0.386438
90,0.394778
100,0.376901


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🏃 View run sft_qwen_gsm8k at: http://mlflow:5000/#/experiments/910887090828468506/runs/7a80854791b64a259c4db5817ac04b70
🧪 View experiment at: http://mlflow:5000/#/experiments/910887090828468506


TrainOutput(global_step=1404, training_loss=0.31879600322144663, metrics={'train_runtime': 409.2845, 'train_samples_per_second': 54.776, 'train_steps_per_second': 3.43, 'total_flos': 1.2544986642329856e+16, 'train_loss': 0.31879600322144663})

In [8]:
# Save model
trainer.save_model("/app/models/sft_qwen")
print("Model saved to /app/models/sft_qwen")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /app/models/sft_qwen
